In [1]:
from starter import rag

query = "How does the agentic loop keep calling the model until it stops?"
answer = rag.rag(query)
print(answer)

The loop keeps calling the model by checking whether the response contains any `function_call` items.

- If there is at least one function call, the code runs the tool, appends the tool output to `messages`, and continues the `while True` loop.
- If there are no function calls in the response, it breaks out of the loop.

So the stop condition is: **no function calls this turn**.


In [ ]:
from opentelemetry import trace
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import ConsoleSpanExporter, SimpleSpanProcessor

provider = TracerProvider()
provider.add_span_processor(
    SimpleSpanProcessor(ConsoleSpanExporter())
)
trace.set_tracer_provider(provider)

tracer = trace.get_tracer("llm-zoomcamp")

In [1]:
from rag_helper import RAGBase

In [2]:
class RAGWithMetrics(RAGBase):

    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
    
    def search(self, *args, **kwargs):
        with tracer.start_as_current_span("search"):
            return super().search(*args, **kwargs)
    
    
    def rag(self, query):
        with tracer.start_as_current_span("rag"):
            return super().rag(query)
    
    def llm(self, prompt):
        with tracer.start_as_current_span("llm") as span:
            response = super().llm(prompt)
            usage = response.usage
            span.set_attribute("input_tokens", usage.input_tokens)
            span.set_attribute("output_tokens", usage.output_tokens)
            return response

In [3]:
from openai import OpenAI
from starter import index

In [4]:
client = OpenAI()

In [5]:
rag = RAGWithMetrics(index=index, llm_client=client)

In [8]:
q = 'How does the agentic loop keep calling the model until it stops?'
answer = rag.rag(q)
print(answer)

{
    "name": "search",
    "context": {
        "trace_id": "0x0724b7b62b175d166c103aabdf0aa227",
        "span_id": "0x183d166203be5245",
        "trace_state": "[]"
    },
    "kind": "SpanKind.INTERNAL",
    "parent_id": "0x4e3d58997b95560a",
    "start_time": "2026-07-23T13:00:33.906439Z",
    "end_time": "2026-07-23T13:00:33.909271Z",
    "status": {
        "status_code": "UNSET"
    },
    "attributes": {},
    "events": [],
    "links": [],
    "resource": {
        "attributes": {
            "telemetry.sdk.language": "python",
            "telemetry.sdk.name": "opentelemetry",
            "telemetry.sdk.version": "1.42.1",
            "service.name": "unknown_service"
        },
        "schema_url": ""
    }
}
{
    "name": "llm",
    "context": {
        "trace_id": "0x0724b7b62b175d166c103aabdf0aa227",
        "span_id": "0x842f3a23e34f3064",
        "trace_state": "[]"
    },
    "kind": "SpanKind.INTERNAL",
    "parent_id": "0x4e3d58997b95560a",
    "start_time": "2026-

In [6]:
import sqlite3
from opentelemetry.sdk.trace.export import SpanExporter, SpanExportResult


class SQLiteSpanExporter(SpanExporter):

    def __init__(self, db_path="traces.db"):
        self.conn = sqlite3.connect(db_path)
        self.conn.execute("""
            CREATE TABLE IF NOT EXISTS spans (
                name TEXT,
                start_time INTEGER,
                end_time INTEGER,
                input_tokens INTEGER,
                output_tokens INTEGER,
                cost REAL
            )
        """)
        self.conn.commit()

    def export(self, spans):
        for span in spans:
            attrs = dict(span.attributes or {})
            self.conn.execute(
                "INSERT INTO spans VALUES (?, ?, ?, ?, ?, ?)",
                (
                    span.name,
                    span.start_time,
                    span.end_time,
                    attrs.get("input_tokens"),
                    attrs.get("output_tokens"),
                    attrs.get("cost"),
                ),
            )
        self.conn.commit()
        return SpanExportResult.SUCCESS

    def shutdown(self):
        self.conn.close()

    def force_flush(self):
        return True

In [7]:
from opentelemetry import trace
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import ConsoleSpanExporter, SimpleSpanProcessor

provider = TracerProvider()
provider.add_span_processor(
    SimpleSpanProcessor(SQLiteSpanExporter("traces.db"))
)
trace.set_tracer_provider(provider)

tracer = trace.get_tracer("llm-zoomcamp")

In [13]:
q = 'How does the agentic loop keep calling the model until it stops?'
answer = rag.rag(q)
print(answer)

It keeps a `while True` loop that:

1. calls the model,
2. checks whether the response contains any `function_call` items,
3. runs those tools and appends the results to the message history,
4. calls the model again with the updated history.

It stops when the model returns a response with **no function calls**. The code uses a flag like `has_function_calls`; if it stays `False` for that turn, the loop `break`s.


In [15]:
import sqlite3

conn = sqlite3.connect("traces.db")
cursor = conn.cursor()

In [16]:
cursor.execute("SELECT * FROM spans")

rows = cursor.fetchall()

for row in rows:
    print(row)

('search', 1784814016952393404, 1784814016960206557, None, None, None)
('llm', 1784814016971606954, 1784814021178599429, 7111, 122, None)
('rag', 1784814016952301162, 1784814021183006128, None, None, None)
('search', 1784819110035581552, 1784819110045983927, None, None, None)
('llm', 1784819110052186394, 1784819114828429640, 7111, 98, None)
('rag', 1784819110035492736, 1784819114835463479, None, None, None)
('search', 1784819765131964781, 1784819765135569542, None, None, None)
('llm', 1784819765142355281, 1784819772661965032, 7111, 105, None)
('rag', 1784819765131901393, 1784819772665410860, None, None, None)
('search', 1784819773970215222, 1784819773975441941, None, None, None)
('llm', 1784819773990929589, 1784819776036117845, 7111, 102, None)
('rag', 1784819773970138288, 1784819776040927633, None, None, None)


In [17]:
cursor.execute('''SELECT name, start_time, end_time, 
                  ROUND((end_time - start_time) / 1000000.0, 3) 
                  AS duration_ms FROM spans;''')

rows = cursor.fetchall()

for row in rows:
    print(row)

('search', 1784814016952393404, 1784814016960206557, 7.813)
('llm', 1784814016971606954, 1784814021178599429, 4206.992)
('rag', 1784814016952301162, 1784814021183006128, 4230.705)
('search', 1784819110035581552, 1784819110045983927, 10.402)
('llm', 1784819110052186394, 1784819114828429640, 4776.243)
('rag', 1784819110035492736, 1784819114835463479, 4799.971)
('search', 1784819765131964781, 1784819765135569542, 3.605)
('llm', 1784819765142355281, 1784819772661965032, 7519.61)
('rag', 1784819765131901393, 1784819772665410860, 7533.509)
('search', 1784819773970215222, 1784819773975441941, 5.227)
('llm', 1784819773990929589, 1784819776036117845, 2045.188)
('rag', 1784819773970138288, 1784819776040927633, 2070.789)


In [18]:
import pandas as pd

In [19]:
df = pd.read_sql_query("SELECT * FROM spans", conn)

In [20]:
df

,name,start_time,end_time,input_tokens,output_tokens,cost
0,search,1784814016952393404,1784814016960206557,NaN,NaN,None
1,llm,1784814016971606954,1784814021178599429,7111.0,122.0,None
2,rag,1784814016952301162,1784814021183006128,NaN,NaN,None
3,search,1784819110035581552,1784819110045983927,NaN,NaN,None
4,llm,1784819110052186394,1784819114828429640,7111.0,98.0,None
5,rag,1784819110035492736,1784819114835463479,NaN,NaN,None
6,search,1784819765131964781,1784819765135569542,NaN,NaN,None
7,llm,1784819765142355281,1784819772661965032,7111.0,105.0,None
8,rag,1784819765131901393,1784819772665410860,NaN,NaN,None
9,search,1784819773970215222,1784819773975441941,NaN,NaN,None


In [21]:
df[df.name=='llm']

,name,start_time,end_time,input_tokens,output_tokens,cost
1,llm,1784814016971606954,1784814021178599429,7111.0,122.0,None
4,llm,1784819110052186394,1784819114828429640,7111.0,98.0,None
7,llm,1784819765142355281,1784819772661965032,7111.0,105.0,None
10,llm,1784819773990929589,1784819776036117845,7111.0,102.0,None


In [22]:
df[df.name=='llm'].input_tokens.var()

np.float64(0.0)